In [35]:
# !pip install xgboost==3.0.5

In [36]:
import xgboost
print(xgboost.__version__)

3.0.5


In [37]:
import pandas as pd
df=pd.read_csv('feature_engineered_data.csv')

In [38]:
df.columns

Index(['loan_amnt', 'term', 'int_rate', 'installment', 'grade', 'emp_length',
       'home_ownership', 'annual_inc', 'verification_status', 'dti',
       'delinq_2yrs', 'inq_last_6mths', 'open_acc', 'revol_util', 'total_acc',
       'loan_status', 'fico_score', 'loan_to_income', 'installment_to_income',
       'loan_to_installment', 'dti_to_income', 'fico_category',
       'fico_rate_mismatch', 'inq_to_open_acc', 'delinq_rate',
       'acc_utilization', 'high_revol_util', 'high_int_rate',
       'rate_vs_grade_avg', 'emp_unknown', 'income_per_emp_year',
       'stable_employment', 'risk_score', 'high_risk', 'term_rate_interaction',
       'log_annual_inc', 'log_loan_amnt'],
      dtype='object')

In [39]:
df.shape

(1345310, 37)

In [40]:
df['emp_length'].value_counts()

emp_length
10.0    442199
6.0     141244
2.0     121743
0.0     108061
3.0     107597
1.0      88494
5.0      84154
4.0      80556
8.0      60701
7.0      59624
9.0      50937
Name: count, dtype: int64

In [41]:
df['loan_status']

0          0
1          0
2          0
3          0
4          0
          ..
1345305    0
1345306    0
1345307    1
1345308    0
1345309    1
Name: loan_status, Length: 1345310, dtype: int64

In [42]:
from sklearn.model_selection import train_test_split

X = df.drop("loan_status", axis=1)
y = df["loan_status"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(1076248, 36)
(269062, 36)


In [43]:
X_train.columns

Index(['loan_amnt', 'term', 'int_rate', 'installment', 'grade', 'emp_length',
       'home_ownership', 'annual_inc', 'verification_status', 'dti',
       'delinq_2yrs', 'inq_last_6mths', 'open_acc', 'revol_util', 'total_acc',
       'fico_score', 'loan_to_income', 'installment_to_income',
       'loan_to_installment', 'dti_to_income', 'fico_category',
       'fico_rate_mismatch', 'inq_to_open_acc', 'delinq_rate',
       'acc_utilization', 'high_revol_util', 'high_int_rate',
       'rate_vs_grade_avg', 'emp_unknown', 'income_per_emp_year',
       'stable_employment', 'risk_score', 'high_risk', 'term_rate_interaction',
       'log_annual_inc', 'log_loan_amnt'],
      dtype='object')

In [44]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
X_train['fico_category'] = le.fit_transform(X_train['fico_category'].astype(str))
X_test['fico_category'] = le.transform(X_test['fico_category'].astype(str))

# Verify
print(X_train['fico_category'].unique())  # Should be numbers now
print(X_train.dtypes[X_train.dtypes == 'object'])

[2 3 1 0]
Series([], dtype: object)


In [45]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(
    sampling_strategy=0.6,   # minority will be 60% of majority
    random_state=42
)

X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("After SMOTE:")
print("X_train_sm shape:", X_train_sm.shape)
print("y_train_sm value counts:\n", y_train_sm.value_counts())

After SMOTE:
X_train_sm shape: (1378241, 36)
y_train_sm value counts:
 loan_status
0    861401
1    516840
Name: count, dtype: int64


In [46]:
pd.set_option("display.max_columns",None)
df

,loan_amnt,term,int_rate,installment,grade,emp_length,home_ownership,annual_inc,verification_status,dti,delinq_2yrs,inq_last_6mths,open_acc,revol_util,total_acc,loan_status,fico_score,loan_to_income,installment_to_income,loan_to_installment,dti_to_income,fico_category,fico_rate_mismatch,inq_to_open_acc,delinq_rate,acc_utilization,high_revol_util,high_int_rate,rate_vs_grade_avg,emp_unknown,income_per_emp_year,stable_employment,risk_score,high_risk,term_rate_interaction,log_annual_inc,log_loan_amnt
0,3600.0,36,13.99,123.03,3,10.0,1,55000.0,0,5.91,0.0,1.0,7.0,29.7,13.0,0,677,0.750242,11.271534,29.261156,325050.00,Good,0.020665,0.125000,0.000000,0.500000,0,0,-0.031234,0,5000.000000,1,6.170,0,503.64,10.915107,8.188967
1,24700.0,36,11.99,820.28,3,10.0,1,65000.0,0,16.06,1.0,4.0,22.0,19.2,38.0,0,717,0.912692,74.018075,30.111669,1043900.00,Good,0.016722,0.173913,0.025641,0.564103,0,0,-2.031234,0,5909.090909,1,9.415,0,431.64,11.082158,10.114599
2,20000.0,60,10.78,432.66,2,10.0,1,63000.0,0,10.78,0.0,0.0,6.0,56.2,18.0,0,697,0.896174,39.151541,46.225674,679140.00,Good,0.015466,0.000000,0.000000,0.315789,0,0,0.100871,0,5727.272727,1,6.468,0,646.80,11.050906,9.903538
3,10400.0,60,22.45,289.91,6,3.0,1,104433.0,1,25.37,1.0,3.0,12.0,64.5,35.0,0,697,0.800399,25.086726,35.873202,2649465.21,Good,0.032209,0.230769,0.027778,0.333333,0,1,-2.484785,0,26108.250000,0,15.146,0,1347.00,11.556311,9.249657
4,11950.0,36,13.44,405.18,3,4.0,0,34000.0,1,10.20,0.0,0.0,5.0,68.4,6.0,0,692,0.899793,38.832122,29.493065,346800.00,Good,0.019422,0.000000,0.000000,0.714286,0,0,-0.581234,0,6800.000000,0,7.092,0,483.84,10.434145,9.388570
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1345305,18000.0,60,9.49,377.95,2,5.0,2,130000.0,0,20.59,0.0,1.0,17.0,34.0,39.0,0,737,0.832096,32.096854,47.625347,2676700.00,Good,0.012877,0.055556,0.000000,0.425000,0,0,-1.189129,0,21666.666667,1,9.224,0,569.40,11.775297,9.798183
1345306,29400.0,60,13.99,683.94,3,9.0,1,180792.0,0,22.03,0.0,1.0,16.0,85.2,32.0,0,707,0.849954,56.500115,42.986227,3982847.76,Good,0.019788,0.058824,0.000000,0.484848,1,0,-0.031234,0,18079.200000,1,11.006,0,839.40,12.105108,10.288784
1345307,32000.0,60,14.49,752.74,3,3.0,1,157000.0,1,10.34,0.0,0.0,14.0,27.4,18.0,1,737,0.867061,62.917045,42.511359,1623380.00,Good,0.019661,0.000000,0.000000,0.736842,0,0,0.468766,0,39250.000000,0,7.449,0,869.40,11.964007,10.373522
1345308,16000.0,60,12.79,362.34,3,10.0,0,150000.0,0,12.25,0.0,0.0,12.0,55.0,28.0,0,667,0.812224,30.401739,44.157421,1837500.00,Fair,0.019175,0.000000,0.000000,0.413793,0,0,-1.231234,0,13636.363636,1,7.512,0,767.40,11.918397,9.680406


In [47]:
sample_size = 400000

X_train_sample = X_train_sm.sample(n=sample_size, random_state=42)
y_train_sample = y_train_sm.loc[X_train_sample.index]

In [48]:
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV

xgb = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    tree_method='hist',
    random_state=42
)

param_grid = {
    "n_estimators": [300,400,500],
    "max_depth": [4,5,6,7],
    "learning_rate": [0.03,0.05,0.07],
    "subsample": [0.7,0.8,1],
    "colsample_bytree": [0.7,0.8,1],
    "scale_pos_weight": [2,3,4]
}

random_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_grid,
    n_iter=15,
    scoring='recall',
    cv=3,
    verbose=2,
    n_jobs=-1,
    random_state=42
)

In [49]:
random_search.fit(X_train_sample, y_train_sample)

Fitting 3 folds for each of 15 candidates, totalling 45 fits


RandomizedSearchCV(cv=3,
                   estimator=XGBClassifier(base_score=None, booster=None,
                                           callbacks=None,
                                           colsample_bylevel=None,
                                           colsample_bynode=None,
                                           colsample_bytree=None, device=None,
                                           early_stopping_rounds=None,
                                           enable_categorical=False,
                                           eval_metric='logloss',
                                           feature_types=None,
                                           feature_weights=None, gamma=None,
                                           grow_policy=None,
                                           importance_type=None,
                                           interaction_cons...
                                           monotone_constraints=None,
                                           multi_strategy=None,
                                           n_estimators=None, n_jobs=None,
                                           num_parallel_tree=None, ...),
                   n_iter=15, n_jobs=-1,
                   param_distributions={'colsample_bytree': [0.7, 0.8, 1],
                                        'learning_rate': [0.03, 0.05, 0.07],
                                        'max_depth': [4, 5, 6, 7],
                                        'n_estimators': [300, 400, 500],
                                        'scale_pos_weight': [2, 3, 4],
                                        'subsample': [0.7, 0.8, 1]},
                   random_state=42, scoring='recall', verbose=2)

In [50]:
best_params = random_search.best_params_

final_model = XGBClassifier(**best_params)

final_model.fit(X_train_sm, y_train_sm)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.7, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.03, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=400,
              n_jobs=None, num_parallel_tree=None, ...)

In [51]:
best_params

{'subsample': 0.8,
 'scale_pos_weight': 4,
 'n_estimators': 400,
 'max_depth': 6,
 'learning_rate': 0.03,
 'colsample_bytree': 0.7}

{'subsample': 0.8,
 'scale_pos_weight': 4,
 'n_estimators': 400,
 'max_depth': 6,
 'learning_rate': 0.03,
 'colsample_bytree': 0.7}

In [52]:
from sklearn.metrics import classification_report, roc_auc_score
pred = final_model.predict(X_test)
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.89      0.60      0.72    215350
           1       0.31      0.71      0.43     53712

    accuracy                           0.62    269062
   macro avg       0.60      0.66      0.57    269062
weighted avg       0.78      0.62      0.66    269062



In [53]:
# import pickle

# # Save the trained model
# with open("loan_default_model.pkl", "wb") as file:
#     pickle.dump(final_model, file)

In [54]:

final_model.save_model('loan_default_model.json')
print("Model saved successfully!")
final_model.load_model('loan_default_model.json')
print("Model loaded successfully!")

Model saved successfully!
Model loaded successfully!


In [55]:
# import pandas as pd
# from sklearn.base import BaseEstimator, TransformerMixin

# class CustomEncoder(BaseEstimator, TransformerMixin):

#     def fit(self, X, y=None):
#         return self

#     def transform(self, X):

#         X = X.copy()

#         home_map = {
#             "RENT":0,
#             "MORTGAGE":1,
#             "OWN":2,
#             "OTHER":3
#         }

#         grade_map = {
#             "A":1,"B":2,"C":3,"D":4,"E":5,"F":6,"G":7
#         }

#         verification_map = {
#             "Not Verified":0,
#             "Source Verified":1,
#             "Verified":2
#         }

#         emp_map = {
#             "Unknown":-1,
#             "< 1 year":0,
#             "1 year":1,
#             "2 years":2,
#             "3 years":3,
#             "4 years":4,
#             "5 years":5,
#             "6 years":6,
#             "7 years":7,
#             "8 years":8,
#             "9 years":9,
#             "10+ years":10
#         }

#         X["home_ownership"] = X["home_ownership"].map(home_map)
#         X["grade"] = X["grade"].map(grade_map)
#         X["verification_status"] = X["verification_status"].map(verification_map)
#         X["emp_length"] = X["emp_length"].map(emp_map)

#         return X

In [56]:
# from sklearn.pipeline import Pipeline
# from imblearn.over_sampling import SMOTE
# from xgboost import XGBClassifier
# import joblib

# def train_model(X_train, y_train):

#     pipeline = Pipeline([

#         ("encoding", CustomEncoder()),

#         ("smote", SMOTE(
#             sampling_strategy=0.6,
#             random_state=42
#         )),

#         ("model", XGBClassifier(
#             n_estimators=400,
#             max_depth=6,
#             learning_rate=0.03,
#             subsample=0.8,
#             colsample_bytree=0.7,
#             scale_pos_weight=4,
#             objective="binary:logistic",
#             eval_metric="logloss"
#         ))

#     ])

#     pipeline.fit(X_train, y_train)

#     joblib.dump(pipeline, "artifacts/model.pkl")

#     return pipeline

In [57]:
# import pandas as pd
# import joblib

# class PredictPipeline:

#     def __init__(self):

#         self.model = joblib.load("artifacts/model.pkl")

#     def predict(self, data):

#         df = pd.DataFrame(data)

#         prediction = self.model.predict(df)

#         return prediction

In [58]:
X_train_sm

,loan_amnt,term,int_rate,installment,grade,emp_length,home_ownership,annual_inc,verification_status,dti,delinq_2yrs,inq_last_6mths,open_acc,revol_util,total_acc,fico_score,loan_to_income,installment_to_income,loan_to_installment,dti_to_income,fico_category,fico_rate_mismatch,inq_to_open_acc,delinq_rate,acc_utilization,high_revol_util,high_int_rate,rate_vs_grade_avg,emp_unknown,income_per_emp_year,stable_employment,risk_score,high_risk,term_rate_interaction,log_annual_inc,log_loan_amnt
0,12000.000000,60,9.990000,254.910000,2,7.000000,0,104000.000000,0,7.730000,0.000000,0.000000,10.000000,64.500000,34.000000,682,0.813073,22.066011,47.075438,8.039200e+05,2,0.014648,0.000000,0.000000,0.285714,0,0,-0.689129,0,13000.000000,1,5.316000,0,599.400000,11.552156,9.392745
1,17000.000000,36,9.170000,541.950000,2,2.000000,0,42000.000000,0,20.600000,0.000000,0.000000,17.000000,11.800000,28.000000,767,0.915042,50.909080,31.368207,8.652000e+05,3,0.011956,0.000000,0.000000,0.586207,0,0,-1.509129,0,14000.000000,0,8.931000,0,330.120000,10.645449,9.741027
2,10000.000000,36,6.240000,305.310000,1,9.000000,2,175000.000000,1,12.370000,0.000000,1.000000,15.000000,91.000000,23.000000,682,0.762924,25.289610,32.753595,2.164750e+06,2,0.009150,0.062500,0.000000,0.625000,1,0,-0.873039,0,17500.000000,1,5.783000,0,224.640000,12.072547,9.210440
3,8000.000000,36,14.030000,273.540000,3,8.000000,0,38000.000000,1,22.710000,0.000000,0.000000,5.000000,43.600000,10.000000,717,0.852253,25.939351,29.246180,8.629800e+05,2,0.019568,0.000000,0.000000,0.454545,0,0,0.008766,0,4222.222222,1,11.022000,0,505.080000,10.545368,8.987322
4,1900.000000,36,12.840000,63.880000,3,0.000000,1,34320.000000,1,23.290000,0.000000,2.000000,9.000000,54.900000,10.000000,672,0.722950,6.116716,29.743269,7.993128e+05,2,0.019107,0.200000,0.000000,0.818182,0,0,-1.181234,0,34320.000000,0,11.239000,0,462.240000,10.443513,7.550135
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1378236,14854.899318,42,24.179135,540.132528,5,10.000000,1,47542.676549,2,15.562154,0.274497,0.274497,7.646980,57.413624,10.646980,685,0.891987,50.148496,28.713864,7.398382e+05,1,0.035206,0.021115,0.017156,0.641174,0,1,1.224339,0,4322.061504,1,12.032185,0,988.109068,10.769379,9.606134
1378237,9954.090408,45,20.889806,328.727697,4,10.000000,0,55249.946170,1,25.835856,0.583423,2.000000,16.584500,34.051776,44.087191,669,0.842972,30.105792,30.737103,1.427393e+06,1,0.031189,0.134048,0.009261,0.391195,0,1,1.744966,0,5022.722379,1,14.534383,0,973.485331,10.919626,9.204961
1378238,26353.245581,36,27.085942,1082.101539,6,5.577371,0,62113.143083,2,44.642231,0.000000,0.633943,13.592343,23.917293,28.267886,671,0.922153,98.138779,24.505539,2.762003e+06,1,0.040421,0.022641,0.000000,0.449482,0,1,1.474059,0,9718.563388,0,21.645240,0,975.093895,11.034690,10.174788
1378239,8529.517539,36,10.973792,277.895883,2,10.000000,1,53960.368900,0,26.234866,0.000000,0.000000,8.000000,73.944721,21.549196,711,0.829248,25.497967,30.571150,1.415097e+06,2,0.015371,0.000000,0.000000,0.359213,0,0,0.294663,0,4905.488082,1,11.162597,0,395.056505,10.895830,9.035633


In [59]:
final_model.predict_proba(X_test)

array([[0.6711161 , 0.3288839 ],
       [0.60116553, 0.39883444],
       [0.7193426 , 0.28065744],
       ...,
       [0.46783465, 0.53216535],
       [0.25659162, 0.7434084 ],
       [0.7611907 , 0.23880929]], dtype=float32)